In [1]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정 (Mac/Linux)
plt.rcParams['font.family'] = 'AppleGothic'  # Windows: 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 데이터 로드
def load_jsonl(filepath):
    """JSONL 파일을 DataFrame으로 로드"""
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return pd.DataFrame(data)

# 데이터 로드
df = load_jsonl('/data/ephemeral/home/T8161/game_data/extracted/steam_games_info.jsonl')

print("="*80)
print("📊 1. 데이터 품질 & 기본 통계 분석")
print("="*80)

# 1-1. 기본 정보
print(f"\n✅ 전체 데이터 수: {len(df):,}개")
print(f"✅ 컬럼 수: {len(df.columns)}개")
print(f"\n컬럼 목록:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")

# 1-2. 데이터 타입 확인
print("\n" + "="*80)
print("📋 데이터 타입")
print("="*80)
print(df.dtypes)

# 1-3. 결측치 분석 (중요 필드 중심)
print("\n" + "="*80)
print("🔍 결측치 분석 (추천 시스템 핵심 필드)")
print("="*80)

important_fields = [
    'name', 'name_en', 'short_description_en', 'short_description_kr',
    'tags_en', 'genres_en', 'genres_kr', 'categories_en',
    'price_int', 'is_free', 'is_korean_supported',
    'metacritic', 'recommendations_total'
]

missing_analysis = []
for field in important_fields:
    if field in df.columns:
        total = len(df)
        missing = df[field].isna().sum()
        
        # 리스트/딕셔너리 타입의 경우 빈 값도 체크
        if df[field].dtype == 'object':
            try:
                empty = df[field].apply(lambda x: x is None or (isinstance(x, (list, dict)) and len(x) == 0)).sum()
            except:
                empty = 0
        else:
            empty = 0
        
        missing_pct = (missing / total) * 100
        empty_pct = (empty / total) * 100
        
        missing_analysis.append({
            'Field': field,
            'Missing': missing,
            'Missing %': f"{missing_pct:.2f}%",
            'Empty': empty,
            'Empty %': f"{empty_pct:.2f}%",
            'Total Invalid': missing + empty,
            'Invalid %': f"{((missing + empty) / total) * 100:.2f}%"
        })

missing_df = pd.DataFrame(missing_analysis)
print(missing_df.to_string(index=False))

# 1-4. price_int 특수값 분석
print("\n" + "="*80)
print("💰 가격 데이터 품질 분석")
print("="*80)

price_analysis = {
    '무료 (0)': (df['price_int'] == 0).sum(),
    '정보없음 (-1)': (df['price_int'] == -1).sum(),
    '에러 (-2)': (df['price_int'] == -2).sum(),
    '유료 (>0)': (df['price_int'] > 0).sum()
}

for category, count in price_analysis.items():
    pct = (count / len(df)) * 100
    print(f"  {category:20s}: {count:6,}개 ({pct:5.2f}%)")

# 1-5. 기본 통계량
print("\n" + "="*80)
print("📈 기본 분포 통계")
print("="*80)

# 타입별 분포
print("\n[게임 타입 분포]")
type_dist = df['type'].value_counts()
for game_type, count in type_dist.items():
    pct = (count / len(df)) * 100
    print(f"  {game_type:15s}: {count:6,}개 ({pct:5.2f}%)")

# 한국 지역 가용성
print("\n[한국 스토어 접속 가능 여부]")
kr_available = df['is_available_in_kr'].value_counts()
for status, count in kr_available.items():
    pct = (count / len(df)) * 100
    print(f"  {'가능' if status else '불가능':10s}: {count:6,}개 ({pct:5.2f}%)")

# 한국어 지원
print("\n[한국어 지원 현황]")
kr_sub = (df['is_korean_supported'] == True).sum()
kr_dub = (df['is_korean_dubbed'] == True).sum()
print(f"  자막 지원: {kr_sub:6,}개 ({(kr_sub/len(df))*100:5.2f}%)")
print(f"  더빙 지원: {kr_dub:6,}개 ({(kr_dub/len(df))*100:5.2f}%)")

# 플랫폼 지원
print("\n[플랫폼 지원 현황]")
platform_counts = {
    'Windows': 0,
    'Mac': 0,
    'Linux': 0
}

for idx, row in df.iterrows():
    if isinstance(row['platforms'], dict):
        if row['platforms'].get('windows', False):
            platform_counts['Windows'] += 1
        if row['platforms'].get('mac', False):
            platform_counts['Mac'] += 1
        if row['platforms'].get('linux', False):
            platform_counts['Linux'] += 1

for platform, count in platform_counts.items():
    pct = (count / len(df)) * 100
    print(f"  {platform:10s}: {count:6,}개 ({pct:5.2f}%)")

# 1-6. 가격 분포 (유료 게임만)
print("\n" + "="*80)
print("💵 유료 게임 가격 분포")
print("="*80)

paid_games = df[df['price_int'] > 0]['price_int']
if len(paid_games) > 0:
    print(f"\n  평균 가격: {paid_games.mean():,.0f} KRW")
    print(f"  중앙값:   {paid_games.median():,.0f} KRW")
    print(f"  최소 가격: {paid_games.min():,.0f} KRW")
    print(f"  최대 가격: {paid_games.max():,.0f} KRW")
    print(f"  표준편차: {paid_games.std():,.0f} KRW")
    
    print(f"\n  분위수:")
    print(f"    25%: {paid_games.quantile(0.25):,.0f} KRW")
    print(f"    50%: {paid_games.quantile(0.50):,.0f} KRW")
    print(f"    75%: {paid_games.quantile(0.75):,.0f} KRW")
    print(f"    90%: {paid_games.quantile(0.90):,.0f} KRW")
    print(f"    95%: {paid_games.quantile(0.95):,.0f} KRW")

# 1-7. 출시 연도 분석
print("\n" + "="*80)
print("📅 출시 연도 분석")
print("="*80)

def extract_year(date_str):
    """출시일에서 연도 추출"""
    if pd.isna(date_str) or not isinstance(date_str, str):
        return None
    try:
        # 다양한 날짜 형식 처리
        for fmt in ['%Y-%m-%d', '%d %b, %Y', '%b %d, %Y', '%Y']:
            try:
                return datetime.strptime(date_str, fmt).year
            except:
                continue
        # 연도만 있는 경우
        if date_str.isdigit() and len(date_str) == 4:
            return int(date_str)
        return None
    except:
        return None

df['release_year'] = df['release_date'].apply(extract_year)
year_dist = df['release_year'].value_counts().sort_index()

print(f"\n  출시 연도 정보 있음: {df['release_year'].notna().sum():,}개")
print(f"  출시 연도 정보 없음: {df['release_year'].isna().sum():,}개")

if len(year_dist) > 0:
    print(f"\n  최초 출시: {year_dist.index.min()}년")
    print(f"  최근 출시: {year_dist.index.max()}년")
    print(f"\n  연도별 상위 5개:")
    for year, count in year_dist.nlargest(5).items():
        print(f"    {year}년: {count:,}개")

# 1-8. 연령 등급 분석
print("\n" + "="*80)
print("🔞 연령 등급 분석")
print("="*80)

age_dist = df['age_rating'].value_counts().sort_index()
for age, count in age_dist.items():
    pct = (count / len(df)) * 100
    age_label = "전체이용가/정보없음" if age == 0 else f"{age}세 이용가"
    print(f"  {age_label:20s}: {count:6,}개 ({pct:5.2f}%)")

print("\n✅ 1단계 분석 완료!")
print("="*80)

📊 1. 데이터 품질 & 기본 통계 분석

✅ 전체 데이터 수: 36,666개
✅ 컬럼 수: 32개

컬럼 목록:
   1. appid
   2. name
   3. type
   4. is_available_in_kr
   5. is_free
   6. price_int
   7. price_currency
   8. release_date
   9. age_rating
  10. dlc_count
  11. supported_languages
  12. audio_languages
  13. is_korean_supported
  14. is_korean_dubbed
  15. platforms
  16. specs
  17. short_description_kr
  18. genres_kr
  19. categories_kr
  20. developers
  21. publishers
  22. name_en
  23. short_description_en
  24. genres_en
  25. categories_en
  26. tags_en
  27. metacritic
  28. recommendations_total
  29. header_image
  30. screenshots_thumbnail
  31. screenshots_full
  32. movies

📋 데이터 타입
appid                    object
name                     object
type                     object
is_available_in_kr         bool
is_free                    bool
price_int                 int64
price_currency           object
release_date             object
age_rating               object
dlc_count                 int64
sup

In [2]:
print("\n\n")
print("="*80)
print("🏷️ 2. 태그/장르 분석 (추천 알고리즘 핵심)")
print("="*80)

from itertools import combinations
from collections import defaultdict

# 2-1. 장르 분석
print("\n" + "="*80)
print("🎮 장르(Genre) 분석")
print("="*80)

# 장르 추출 (영어)
all_genres = []
for genres in df['genres_en']:
    if isinstance(genres, list) and len(genres) > 0:
        all_genres.extend(genres)

genre_counts = Counter(all_genres)
print(f"\n✅ 총 고유 장르 수: {len(genre_counts)}개")
print(f"✅ 장르 정보 있는 게임: {(df['genres_en'].apply(lambda x: isinstance(x, list) and len(x) > 0)).sum():,}개")

print(f"\n📊 Top 20 인기 장르:")
for i, (genre, count) in enumerate(genre_counts.most_common(20), 1):
    pct = (count / len(df)) * 100
    print(f"  {i:2d}. {genre:30s}: {count:6,}개 ({pct:5.2f}%)")

# 2-2. 장르 조합 분석
print("\n" + "="*80)
print("🔗 장르 조합 패턴 분석 (2개 조합)")
print("="*80)

genre_combinations = []
for genres in df['genres_en']:
    if isinstance(genres, list) and len(genres) >= 2:
        # 정렬해서 순서 무관하게 조합 저장
        sorted_genres = sorted(genres)
        for combo in combinations(sorted_genres, 2):
            genre_combinations.append(combo)

combo_counts = Counter(genre_combinations)
print(f"\n✅ 총 고유 장르 조합: {len(combo_counts)}개")
print(f"\n📊 Top 15 인기 장르 조합:")
for i, (combo, count) in enumerate(combo_counts.most_common(15), 1):
    print(f"  {i:2d}. {combo[0]:20s} + {combo[1]:20s}: {count:5,}개")

# 2-3. 태그 분석 (가장 중요!)
print("\n" + "="*80)
print("🏷️ 유저 태그(Tags) 분석 - 추천 시스템 핵심 지표")
print("="*80)

all_tags = []
tag_positions = defaultdict(list)  # 태그별 순위 분포 저장

for idx, tags in enumerate(df['tags_en']):
    if isinstance(tags, list) and len(tags) > 0:
        all_tags.extend(tags)
        # 태그 순위 정보도 저장 (상위 태그일수록 중요도 높음)
        for position, tag in enumerate(tags):
            tag_positions[tag].append(position)

tag_counts = Counter(all_tags)
print(f"\n✅ 총 고유 태그 수: {len(tag_counts)}개")
print(f"✅ 태그 정보 있는 게임: {(df['tags_en'].apply(lambda x: isinstance(x, list) and len(x) > 0)).sum():,}개")
print(f"✅ 게임당 평균 태그 수: {len(all_tags) / len(df):.2f}개")

print(f"\n📊 Top 50 인기 태그 (추천 시스템 필터링 후보):")
for i, (tag, count) in enumerate(tag_counts.most_common(50), 1):
    pct = (count / len(df)) * 100
    avg_position = np.mean(tag_positions[tag]) if tag in tag_positions else 0
    print(f"  {i:2d}. {tag:30s}: {count:6,}개 ({pct:5.2f}%) | 평균순위: {avg_position:.1f}")

# 2-4. 태그 조합 분석 (공출현 패턴)
print("\n" + "="*80)
print("🔗 태그 조합 패턴 분석 (2개 조합 - Top 5 태그 기준)")
print("="*80)

# Top 5 태그로 한정해서 조합 분석 (너무 많아서)
top_5_tags = [tag for tag, _ in tag_counts.most_common(5)]
print(f"\n분석 대상 태그: {', '.join(top_5_tags)}")

tag_combinations = []
for tags in df['tags_en']:
    if isinstance(tags, list) and len(tags) >= 2:
        # Top 5 태그만 필터링
        filtered_tags = [tag for tag in tags if tag in top_5_tags]
        if len(filtered_tags) >= 2:
            sorted_tags = sorted(filtered_tags)
            for combo in combinations(sorted_tags, 2):
                tag_combinations.append(combo)

tag_combo_counts = Counter(tag_combinations)
print(f"\n📊 Top 10 태그 조합:")
for i, (combo, count) in enumerate(tag_combo_counts.most_common(10), 1):
    print(f"  {i:2d}. {combo[0]:20s} + {combo[1]:20s}: {count:5,}개")

# 2-5. 카테고리 분석
print("\n" + "="*80)
print("📁 카테고리(Category) 분석")
print("="*80)

all_categories = []
for categories in df['categories_en']:
    if isinstance(categories, list) and len(categories) > 0:
        all_categories.extend(categories)

category_counts = Counter(all_categories)
print(f"\n✅ 총 고유 카테고리 수: {len(category_counts)}개")
print(f"✅ 카테고리 정보 있는 게임: {(df['categories_en'].apply(lambda x: isinstance(x, list) and len(x) > 0)).sum():,}개")

print(f"\n📊 Top 20 인기 카테고리:")
for i, (category, count) in enumerate(category_counts.most_common(20), 1):
    pct = (count / len(df)) * 100
    print(f"  {i:2d}. {category:40s}: {count:6,}개 ({pct:5.2f}%)")

# 2-6. 멀티플레이어 관련 카테고리 조합
print("\n" + "="*80)
print("👥 멀티플레이어 관련 카테고리 조합")
print("="*80)

multiplayer_keywords = ['Multi-player', 'Co-op', 'Online', 'PvP', 'Multiplayer']
mp_categories = [cat for cat in category_counts.keys() 
                 if any(keyword in cat for keyword in multiplayer_keywords)]

print(f"\n멀티플레이어 관련 카테고리: {len(mp_categories)}개")
for cat in sorted(mp_categories):
    count = category_counts[cat]
    pct = (count / len(df)) * 100
    print(f"  - {cat:40s}: {count:6,}개 ({pct:5.2f}%)")

# 2-7. 장르-태그 상관관계 (주요 장르별 대표 태그)
print("\n" + "="*80)
print("🔍 장르별 대표 태그 분석 (Top 5 장르)")
print("="*80)

top_5_genres = [genre for genre, _ in genre_counts.most_common(5)]

for genre in top_5_genres:
    print(f"\n【{genre}】")
    
    # 해당 장르를 가진 게임들의 태그 수집
    genre_tags = []
    for idx, row in df.iterrows():
        if isinstance(row['genres_en'], list) and genre in row['genres_en']:
            if isinstance(row['tags_en'], list):
                genre_tags.extend(row['tags_en'])
    
    genre_tag_counts = Counter(genre_tags)
    print(f"  Top 10 태그:")
    for i, (tag, count) in enumerate(genre_tag_counts.most_common(10), 1):
        genre_games = (df['genres_en'].apply(lambda x: isinstance(x, list) and genre in x)).sum()
        pct = (count / genre_games) * 100 if genre_games > 0 else 0
        print(f"    {i:2d}. {tag:25s}: {count:5,}개 ({pct:5.1f}%)")

# 2-8. 임베딩 전략을 위한 텍스트 조합 샘플
print("\n" + "="*80)
print("📝 임베딩 전략 샘플 (Description + Tags + Genres 조합)")
print("="*80)

print("\n💡 추천 임베딩 필드 조합:")
print("  1. short_description_en (게임 설명)")
print("  2. tags_en (유저 태그 Top 10)")
print("  3. genres_en (장르)")
print("  4. categories_en (주요 카테고리)")

# 샘플 3개 게임의 임베딩 텍스트 생성 예시
print("\n🎯 임베딩 텍스트 생성 샘플 (상위 3개 게임):")

sample_games = df[df['tags_en'].apply(lambda x: isinstance(x, list) and len(x) > 0)].head(3)

for idx, row in sample_games.iterrows():
    print(f"\n{'='*80}")
    print(f"게임: {row['name']}")
    print(f"{'='*80}")
    
    # 임베딩용 텍스트 조합
    embedding_text_parts = []
    
    # 1. 설명
    if row['short_description_en']:
        embedding_text_parts.append(f"Description: {row['short_description_en']}")
    
    # 2. 장르
    if isinstance(row['genres_en'], list) and len(row['genres_en']) > 0:
        embedding_text_parts.append(f"Genres: {', '.join(row['genres_en'])}")
    
    # 3. 태그 (Top 10만)
    if isinstance(row['tags_en'], list) and len(row['tags_en']) > 0:
        top_tags = row['tags_en'][:10]
        embedding_text_parts.append(f"Tags: {', '.join(top_tags)}")
    
    # 4. 카테고리 (주요 5개만)
    if isinstance(row['categories_en'], list) and len(row['categories_en']) > 0:
        top_cats = row['categories_en'][:5]
        embedding_text_parts.append(f"Categories: {', '.join(top_cats)}")
    
    embedding_text = " | ".join(embedding_text_parts)
    
    print(f"\n임베딩 텍스트 (총 {len(embedding_text)}자):")
    print(f"{embedding_text[:500]}..." if len(embedding_text) > 500 else embedding_text)

print("\n\n✅ 2단계 분석 완료!")
print("="*80)




🏷️ 2. 태그/장르 분석 (추천 알고리즘 핵심)

🎮 장르(Genre) 분석

✅ 총 고유 장르 수: 30개
✅ 장르 정보 있는 게임: 36,307개

📊 Top 20 인기 장르:
   1. Indie                         : 23,861개 (65.08%)
   2. Action                        : 15,099개 (41.18%)
   3. Adventure                     : 14,807개 (40.38%)
   4. Casual                        : 12,895개 (35.17%)
   5. Simulation                    :  8,709개 (23.75%)
   6. RPG                           :  7,788개 (21.24%)
   7. Strategy                      :  7,705개 (21.01%)
   8. Free To Play                  :  2,833개 ( 7.73%)
   9. Early Access                  :  2,525개 ( 6.89%)
  10. Sports                        :  1,594개 ( 4.35%)
  11. Racing                        :  1,223개 ( 3.34%)
  12. Massively Multiplayer         :  1,165개 ( 3.18%)
  13. Utilities                     :    334개 ( 0.91%)
  14. Violent                       :    216개 ( 0.59%)
  15. Design & Illustration         :    211개 ( 0.58%)
  16. Animation & Modeling          :    186개 ( 0.51%)
  17. Gore     

In [4]:
print("\n\n")
print("="*80)
print("⭐ 3. 품질 지표 분석 (메타크리틱 & 추천수)")
print("="*80)

# 3-1. 메타크리틱 점수 분석
print("\n" + "="*80)
print("🎬 메타크리틱(Metacritic) 점수 분석")
print("="*80)

# 메타크리틱 점수가 있는 게임 (0보다 큰 경우)
metacritic_valid = df[df['metacritic'] > 0]
metacritic_missing = df[df['metacritic'] == 0]

print(f"\n✅ 메타크리틱 점수 있음: {len(metacritic_valid):,}개 ({len(metacritic_valid)/len(df)*100:.2f}%)")
print(f"✅ 메타크리틱 점수 없음: {len(metacritic_missing):,}개 ({len(metacritic_missing)/len(df)*100:.2f}%)")

if len(metacritic_valid) > 0:
    print(f"\n📊 메타크리틱 점수 통계:")
    print(f"  평균:   {metacritic_valid['metacritic'].mean():.1f}점")
    print(f"  중앙값: {metacritic_valid['metacritic'].median():.1f}점")
    print(f"  최소:   {metacritic_valid['metacritic'].min()}점")
    print(f"  최대:   {metacritic_valid['metacritic'].max()}점")
    print(f"  표준편차: {metacritic_valid['metacritic'].std():.1f}점")
    
    print(f"\n  분위수:")
    print(f"    25%: {metacritic_valid['metacritic'].quantile(0.25):.0f}점")
    print(f"    50%: {metacritic_valid['metacritic'].quantile(0.50):.0f}점")
    print(f"    75%: {metacritic_valid['metacritic'].quantile(0.75):.0f}점")
    print(f"    90%: {metacritic_valid['metacritic'].quantile(0.90):.0f}점")
    
    # 점수 등급별 분포
    print(f"\n📊 점수 등급별 분포:")
    score_ranges = [
        (90, 100, "명작 (90+)"),
        (80, 89, "우수 (80-89)"),
        (70, 79, "양호 (70-79)"),
        (60, 69, "평범 (60-69)"),
        (0, 59, "미흡 (60점 미만)")
    ]
    
    for min_score, max_score, label in score_ranges:
        count = ((metacritic_valid['metacritic'] >= min_score) & 
                 (metacritic_valid['metacritic'] <= max_score)).sum()
        pct = (count / len(metacritic_valid)) * 100
        print(f"  {label:20s}: {count:5,}개 ({pct:5.2f}%)")

# 3-2. 추천수(Recommendations) 분석
print("\n" + "="*80)
print("👍 스팀 추천수(Recommendations) 분석")
print("="*80)

# 추천수가 있는 게임
recs_valid = df[df['recommendations_total'] > 0]
recs_zero = df[df['recommendations_total'] == 0]

print(f"\n✅ 추천수 있음: {len(recs_valid):,}개 ({len(recs_valid)/len(df)*100:.2f}%)")
print(f"✅ 추천수 0개:  {len(recs_zero):,}개 ({len(recs_zero)/len(df)*100:.2f}%)")

if len(recs_valid) > 0:
    print(f"\n📊 추천수 통계 (추천이 있는 게임만):")
    print(f"  평균:   {recs_valid['recommendations_total'].mean():,.0f}개")
    print(f"  중앙값: {recs_valid['recommendations_total'].median():,.0f}개")
    print(f"  최소:   {recs_valid['recommendations_total'].min():,}개")
    print(f"  최대:   {recs_valid['recommendations_total'].max():,}개")
    
    print(f"\n  분위수:")
    print(f"    25%: {recs_valid['recommendations_total'].quantile(0.25):,.0f}개")
    print(f"    50%: {recs_valid['recommendations_total'].quantile(0.50):,.0f}개")
    print(f"    75%: {recs_valid['recommendations_total'].quantile(0.75):,.0f}개")
    print(f"    90%: {recs_valid['recommendations_total'].quantile(0.90):,.0f}개")
    print(f"    95%: {recs_valid['recommendations_total'].quantile(0.95):,.0f}개")
    print(f"    99%: {recs_valid['recommendations_total'].quantile(0.99):,.0f}개")
    
    # 인기도 등급별 분포
    print(f"\n📊 인기도 등급별 분포:")
    popularity_ranges = [
        (100000, float('inf'), "초대박 (100K+)"),
        (50000, 99999, "대박 (50K-100K)"),
        (10000, 49999, "매우 인기 (10K-50K)"),
        (5000, 9999, "인기 (5K-10K)"),
        (1000, 4999, "준인기 (1K-5K)"),
        (100, 999, "보통 (100-1K)"),
        (1, 99, "신작/니치 (1-100)")
    ]
    
    for min_rec, max_rec, label in popularity_ranges:
        count = ((recs_valid['recommendations_total'] >= min_rec) & 
                 (recs_valid['recommendations_total'] <= max_rec)).sum()
        pct = (count / len(recs_valid)) * 100
        print(f"  {label:25s}: {count:6,}개 ({pct:5.2f}%)")

# 3-3. 메타크리틱 vs 추천수 상관관계
print("\n" + "="*80)
print("🔗 메타크리틱 점수 vs 추천수 상관관계")
print("="*80)

# 둘 다 있는 게임만
both_valid = df[(df['metacritic'] > 0) & (df['recommendations_total'] > 0)]

if len(both_valid) > 100:  # 충분한 데이터가 있을 때만
    correlation = both_valid['metacritic'].corr(both_valid['recommendations_total'])
    print(f"\n📊 피어슨 상관계수: {correlation:.3f}")
    
    if correlation > 0.3:
        print("  → 양의 상관관계 (점수가 높을수록 추천 많음)")
    elif correlation < -0.3:
        print("  → 음의 상관관계")
    else:
        print("  → 약한 상관관계 (점수와 인기는 별개)")
    
    # 메타크리틱 점수대별 평균 추천수
    print(f"\n📊 점수대별 평균 추천수:")
    score_bins = [0, 60, 70, 80, 90, 100]
    for i in range(len(score_bins)-1):
        min_s, max_s = score_bins[i], score_bins[i+1]
        subset = both_valid[(both_valid['metacritic'] >= min_s) & 
                           (both_valid['metacritic'] < max_s)]
        if len(subset) > 0:
            avg_recs = subset['recommendations_total'].mean()
            print(f"  {min_s:2d}-{max_s:2d}점: 평균 {avg_recs:8,.0f}개 추천 (게임 {len(subset):,}개)")

# 3-4. 가격대별 평점/인기도 분석
print("\n" + "="*80)
print("💰 가격대별 품질 지표 분석")
print("="*80)

# 유료 게임만
paid_games = df[df['price_int'] > 0].copy()

if len(paid_games) > 0:
    # 가격 구간 정의
    price_bins = [0, 5000, 10000, 20000, 50000, float('inf')]
    price_labels = ['저가 (0-5K)', '중저가 (5K-10K)', '중가 (10K-20K)', 
                    '고가 (20K-50K)', '프리미엄 (50K+)']
    
    paid_games['price_range'] = pd.cut(paid_games['price_int'], 
                                        bins=price_bins, 
                                        labels=price_labels, 
                                        include_lowest=True)
    
    print(f"\n📊 가격대별 게임 수:")
    for label in price_labels:
        count = (paid_games['price_range'] == label).sum()
        pct = (count / len(paid_games)) * 100
        print(f"  {label:20s}: {count:6,}개 ({pct:5.2f}%)")
    
    # 가격대별 메타크리틱 평균
    print(f"\n📊 가격대별 메타크리틱 평균 (점수 있는 게임만):")
    for label in price_labels:
        subset = paid_games[(paid_games['price_range'] == label) & 
                           (paid_games['metacritic'] > 0)]
        if len(subset) > 0:
            avg_meta = subset['metacritic'].mean()
            print(f"  {label:20s}: 평균 {avg_meta:5.1f}점 (게임 {len(subset):,}개)")
    
    # 가격대별 추천수 평균
    print(f"\n📊 가격대별 추천수 평균 (추천 있는 게임만):")
    for label in price_labels:
        subset = paid_games[(paid_games['price_range'] == label) & 
                           (paid_games['recommendations_total'] > 0)]
        if len(subset) > 0:
            avg_recs = subset['recommendations_total'].mean()
            median_recs = subset['recommendations_total'].median()
            print(f"  {label:20s}: 평균 {avg_recs:8,.0f}개, 중앙값 {median_recs:6,.0f}개 (게임 {len(subset):,}개)")

# 3-5. 무료 게임 vs 유료 게임 품질 비교
print("\n" + "="*80)
print("🆓 무료 게임 vs 유료 게임 품질 비교")
print("="*80)

free_games = df[df['is_free'] == True]
paid_games_comp = df[df['price_int'] > 0]

print(f"\n【무료 게임】 (총 {len(free_games):,}개)")

# 메타크리틱
free_with_meta = free_games[free_games['metacritic'] > 0]
if len(free_with_meta) > 0:
    print(f"  메타크리틱: 평균 {free_with_meta['metacritic'].mean():.1f}점 (데이터 {len(free_with_meta):,}개)")
else:
    print(f"  메타크리틱: 데이터 없음")

# 추천수
free_with_recs = free_games[free_games['recommendations_total'] > 0]
if len(free_with_recs) > 0:
    print(f"  추천수: 평균 {free_with_recs['recommendations_total'].mean():,.0f}개, 중앙값 {free_with_recs['recommendations_total'].median():,.0f}개")

print(f"\n【유료 게임】 (총 {len(paid_games_comp):,}개)")

# 메타크리틱
paid_with_meta = paid_games_comp[paid_games_comp['metacritic'] > 0]
if len(paid_with_meta) > 0:
    print(f"  메타크리틱: 평균 {paid_with_meta['metacritic'].mean():.1f}점 (데이터 {len(paid_with_meta):,}개)")

# 추천수
paid_with_recs = paid_games_comp[paid_games_comp['recommendations_total'] > 0]
if len(paid_with_recs) > 0:
    print(f"  추천수: 평균 {paid_with_recs['recommendations_total'].mean():,.0f}개, 중앙값 {paid_with_recs['recommendations_total'].median():,.0f}개")

# 3-6. Top 20 인기 게임 (추천수 기준)
print("\n" + "="*80)
print("🏆 Top 20 최다 추천 게임")
print("="*80)

top_games = df.nlargest(20, 'recommendations_total')
print(f"\n{'순위':4s} {'게임명':50s} {'추천수':>12s} {'메타점수':>8s} {'가격':>10s}")
print("-" * 90)

for idx, (i, row) in enumerate(top_games.iterrows(), 1):
    name = row['name'][:48] if len(row['name']) > 48 else row['name']
    recs = f"{row['recommendations_total']:,}"
    meta = f"{row['metacritic']}" if row['metacritic'] > 0 else "N/A"
    
    if row['is_free']:
        price = "무료"
    elif row['price_int'] > 0:
        price = f"{row['price_int']:,}원"
    else:
        price = "정보없음"
    
    print(f"{idx:2d}.  {name:50s} {recs:>12s} {meta:>8s} {price:>10s}")

# 3-7. Top 20 고평점 게임 (메타크리틱 기준)
print("\n" + "="*80)
print("🌟 Top 20 메타크리틱 고평점 게임")
print("="*80)

top_rated = df[df['metacritic'] > 0].nlargest(20, 'metacritic')
print(f"\n{'순위':4s} {'게임명':50s} {'메타점수':>8s} {'추천수':>12s} {'가격':>10s}")
print("-" * 90)

for idx, (i, row) in enumerate(top_rated.iterrows(), 1):
    name = row['name'][:48] if len(row['name']) > 48 else row['name']
    meta = f"{row['metacritic']}"
    recs = f"{row['recommendations_total']:,}" if row['recommendations_total'] > 0 else "N/A"
    
    if row['is_free']:
        price = "무료"
    elif row['price_int'] > 0:
        price = f"{row['price_int']:,}원"
    else:
        price = "정보없음"
    
    print(f"{idx:2d}.  {name:50s} {meta:>8s} {recs:>12s} {price:>10s}")

print("\n\n✅ 3단계 분석 완료!")
print("="*80)




⭐ 3. 품질 지표 분석 (메타크리틱 & 추천수)

🎬 메타크리틱(Metacritic) 점수 분석

✅ 메타크리틱 점수 있음: 4,517개 (12.32%)
✅ 메타크리틱 점수 없음: 32,149개 (87.68%)

📊 메타크리틱 점수 통계:
  평균:   73.9점
  중앙값: 76.0점
  최소:   6점
  최대:   97점
  표준편차: 10.5점

  분위수:
    25%: 68점
    50%: 76점
    75%: 81점
    90%: 85점

📊 점수 등급별 분포:
  명작 (90+)            :   139개 ( 3.08%)
  우수 (80-89)          : 1,359개 (30.09%)
  양호 (70-79)          : 1,735개 (38.41%)
  평범 (60-69)          :   856개 (18.95%)
  미흡 (60점 미만)         :   428개 ( 9.48%)

👍 스팀 추천수(Recommendations) 분석

✅ 추천수 있음: 20,353개 (55.51%)
✅ 추천수 0개:  16,313개 (44.49%)

📊 추천수 통계 (추천이 있는 게임만):
  평균:   5,541개
  중앙값: 464개
  최소:   101개
  최대:   4,875,891개

  분위수:
    25%: 204개
    50%: 464개
    75%: 1,578개
    90%: 6,377개
    95%: 15,496개
    99%: 90,812개

📊 인기도 등급별 분포:
  초대박 (100K+)              :    185개 ( 0.91%)
  대박 (50K-100K)            :    178개 ( 0.87%)
  매우 인기 (10K-50K)          :  1,092개 ( 5.37%)
  인기 (5K-10K)              :    927개 ( 4.55%)
  준인기 (1K-5K)              :  4,218개 (20.72%)
  보통 (10

In [8]:
print("\n\n")
print("="*80)
print("📝 4. 텍스트 데이터 분석 & 임베딩 전략")
print("="*80)

import re

# 4-1. Description 길이 분석
print("\n" + "="*80)
print("📄 게임 설명(Description) 텍스트 분석")
print("="*80)

# 영어 설명 길이
df['desc_en_length'] = df['short_description_en'].apply(
    lambda x: len(str(x)) if pd.notna(x) else 0
)

# 한국어 설명 길이
df['desc_kr_length'] = df['short_description_kr'].apply(
    lambda x: len(str(x)) if pd.notna(x) else 0
)

print(f"\n【영어 설명 (short_description_en)】")
print(f"  평균 길이:  {df['desc_en_length'].mean():.0f}자")
print(f"  중앙값:    {df['desc_en_length'].median():.0f}자")
print(f"  최소:      {df['desc_en_length'].min()}자")
print(f"  최대:      {df['desc_en_length'].max()}자")
print(f"  표준편차:   {df['desc_en_length'].std():.0f}자")

print(f"\n  분위수:")
print(f"    25%: {df['desc_en_length'].quantile(0.25):.0f}자")
print(f"    50%: {df['desc_en_length'].quantile(0.50):.0f}자")
print(f"    75%: {df['desc_en_length'].quantile(0.75):.0f}자")
print(f"    90%: {df['desc_en_length'].quantile(0.90):.0f}자")

# 길이별 분포
print(f"\n📊 영어 설명 길이 분포:")
length_ranges = [
    (0, 0, "설명 없음"),
    (1, 100, "매우 짧음 (1-100자)"),
    (101, 200, "짧음 (101-200자)"),
    (201, 400, "보통 (201-400자)"),
    (401, 600, "긴 편 (401-600자)"),
    (601, float('inf'), "매우 김 (600자+)")
]

for min_len, max_len, label in length_ranges:
    count = ((df['desc_en_length'] >= min_len) & 
             (df['desc_en_length'] <= max_len)).sum()
    pct = (count / len(df)) * 100
    print(f"  {label:25s}: {count:6,}개 ({pct:5.2f}%)")

print(f"\n【한국어 설명 (short_description_kr)】")
print(f"  평균 길이:  {df['desc_kr_length'].mean():.0f}자")
print(f"  중앙값:    {df['desc_kr_length'].median():.0f}자")

# 한국어 설명 품질
kr_desc_valid = df[df['desc_kr_length'] > 50]  # 50자 이상을 유의미한 설명으로 간주
print(f"\n  유의미한 한국어 설명 (50자 이상): {len(kr_desc_valid):,}개 ({len(kr_desc_valid)/len(df)*100:.2f}%)")

# 4-2. 주요 키워드 분석 (TF-IDF 스타일)
print("\n" + "="*80)
print("🔑 영어 설명 주요 키워드 분석")
print("="*80)

# 모든 설명 합치기
all_descriptions = ' '.join(df['short_description_en'].dropna().astype(str))

# 단어 추출 (알파벳만, 소문자 변환)
words = re.findall(r'\b[a-z]{3,}\b', all_descriptions.lower())  # 3글자 이상만
word_counts = Counter(words)

# 불용어 제거 (간단한 버전)
stopwords = {'the', 'and', 'for', 'you', 'your', 'with', 'from', 'are', 'this', 
             'that', 'will', 'can', 'has', 'have', 'into', 'about', 'through',
             'was', 'were', 'been', 'being', 'but', 'not', 'all', 'more', 'new'}

filtered_words = {word: count for word, count in word_counts.items() 
                  if word not in stopwords}

print(f"\n✅ 총 고유 단어 수: {len(word_counts):,}개")
print(f"✅ 불용어 제거 후: {len(filtered_words):,}개")

print(f"\n📊 Top 50 빈출 키워드 (게임 설명에서):")
for i, (word, count) in enumerate(Counter(filtered_words).most_common(50), 1):
    print(f"  {i:2d}. {word:20s}: {count:7,}회")

# 4-3. 임베딩용 텍스트 조합 전략
print("\n" + "="*80)
print("🎯 최종 임베딩 전략 수립")
print("="*80)

print("""
💡 추천 임베딩 필드 조합:

【방법 1: 기본 조합 (속도 우선)】
  - short_description_en (게임 설명)
  - genres_en (장르, 최대 5개)
  - tags_en (유저 태그, Top 10개)
  → 예상 토큰: 100-200 토큰

【방법 2: 풍부한 조합 (품질 우선)】 ⭐ 추천
  - short_description_en (게임 설명)
  - genres_en (모든 장르)
  - tags_en (Top 15개 태그)
  - categories_en (주요 카테고리 5개)
  → 예상 토큰: 150-300 토큰

【방법 3: 한국어 사용자 특화】
  - short_description_kr (한국어 설명) + short_description_en (영어 설명)
  - genres_kr + genres_en
  - tags_en (영어만 사용, Top 10개)
  → 한국어 검색 시 정확도 향상
""")

# 실제 임베딩 텍스트 생성 함수
def create_embedding_text(row, method='rich'):
    """임베딩용 텍스트 생성"""
    parts = []
    
    # Description
    if method == 'korean' and row['desc_kr_length'] > 50:
        parts.append(f"{row['short_description_kr']}")
    parts.append(f"{row['short_description_en']}")
    
    # Genres
    if isinstance(row['genres_en'], list) and len(row['genres_en']) > 0:
        if method == 'basic':
            genres = row['genres_en'][:5]
        else:
            genres = row['genres_en']
        parts.append(f"Genres: {', '.join(genres)}")
    
    # Tags
    if isinstance(row['tags_en'], list) and len(row['tags_en']) > 0:
        if method == 'basic':
            tags = row['tags_en'][:10]
        elif method == 'rich':
            tags = row['tags_en'][:15]
        else:
            tags = row['tags_en'][:10]
        parts.append(f"Tags: {', '.join(tags)}")
    
    # Categories (rich 모드에만)
    if method == 'rich' and isinstance(row['categories_en'], list) and len(row['categories_en']) > 0:
        cats = row['categories_en'][:5]
        parts.append(f"Features: {', '.join(cats)}")
    
    return " | ".join(parts)

# 샘플 생성
print("\n" + "="*80)
print("📋 임베딩 텍스트 샘플 비교 (상위 3개 게임)")
print("="*80)

sample_df = df.head(3)

for idx, row in sample_df.iterrows():
    print(f"\n{'='*80}")
    print(f"🎮 게임: {row['name']}")
    print(f"{'='*80}")
    
    # 세 가지 방법으로 생성
    for method, method_name in [('basic', '기본'), ('rich', '풍부'), ('korean', '한국어')]:
        text = create_embedding_text(row, method=method)
        print(f"\n[{method_name} 방법] (총 {len(text)}자):")
        if len(text) > 400:
            print(f"{text[:400]}...")
        else:
            print(text)

# 4-4. 임베딩 텍스트 길이 통계
print("\n" + "="*80)
print("📏 임베딩 텍스트 길이 분석")
print("="*80)

# 각 방법별로 길이 계산
df['embed_basic'] = df.apply(lambda row: create_embedding_text(row, 'basic'), axis=1)
df['embed_rich'] = df.apply(lambda row: create_embedding_text(row, 'rich'), axis=1)

df['embed_basic_len'] = df['embed_basic'].str.len()
df['embed_rich_len'] = df['embed_rich'].str.len()

print(f"\n【기본 방법】")
print(f"  평균: {df['embed_basic_len'].mean():.0f}자")
print(f"  중앙값: {df['embed_basic_len'].median():.0f}자")
print(f"  최대: {df['embed_basic_len'].max()}자")

print(f"\n【풍부한 방법】 ⭐")
print(f"  평균: {df['embed_rich_len'].mean():.0f}자")
print(f"  중앙값: {df['embed_rich_len'].median():.0f}자")
print(f"  최대: {df['embed_rich_len'].max()}자")

# 대략적인 토큰 수 추정 (영어 기준 1토큰 ≈ 4자)
print(f"\n💡 예상 토큰 수 (1토큰 ≈ 4자 기준):")
print(f"  기본 방법: 평균 {df['embed_basic_len'].mean()/4:.0f} 토큰")
print(f"  풍부한 방법: 평균 {df['embed_rich_len'].mean()/4:.0f} 토큰")

print("\n\n✅ 4단계 분석 완료!")
print("="*80)




📝 4. 텍스트 데이터 분석 & 임베딩 전략

📄 게임 설명(Description) 텍스트 분석

【영어 설명 (short_description_en)】
  평균 길이:  211자
  중앙값:    226자
  최소:      0자
  최대:      342자
  표준편차:   75자

  분위수:
    25%: 158자
    50%: 226자
    75%: 273자
    90%: 296자

📊 영어 설명 길이 분포:
  설명 없음                    :    177개 ( 0.48%)
  매우 짧음 (1-100자)           :  3,556개 ( 9.70%)
  짧음 (101-200자)            : 10,675개 (29.11%)
  보통 (201-400자)            : 22,258개 (60.70%)
  긴 편 (401-600자)           :      0개 ( 0.00%)
  매우 김 (600자+)             :      0개 ( 0.00%)

【한국어 설명 (short_description_kr)】
  평균 길이:  187자
  중앙값:    184자

  유의미한 한국어 설명 (50자 이상): 35,121개 (95.79%)

🔑 영어 설명 주요 키워드 분석

✅ 총 고유 단어 수: 40,348개
✅ 불용어 제거 후: 40,322개

📊 Top 50 빈출 키워드 (게임 설명에서):
   1. game                :  14,449회
   2. world               :   7,821회
   3. adventure           :   4,088회
   4. quot                :   4,085회
   5. where               :   3,975회
   6. play                :   3,769회
   7. action              :   3,643회
   8. time                : 

In [9]:
print("\n\n")
print("="*80)
print("🎯 5. 추천 시나리오별 필터링 조건 & 최종 요약")
print("="*80)

# 5-1. 가격대별 게임 수 (세분화)
print("\n" + "="*80)
print("💰 가격대별 게임 분포 (필터링 옵션 설계)")
print("="*80)

price_scenarios = [
    (0, 0, "무료"),
    (1, 5000, "5천원 이하"),
    (1, 10000, "1만원 이하"),
    (1, 20000, "2만원 이하"),
    (1, 50000, "5만원 이하"),
    (50001, float('inf'), "5만원 초과")
]

print(f"\n📊 가격 필터 옵션별 게임 수:")
for min_p, max_p, label in price_scenarios:
    count = ((df['price_int'] >= min_p) & (df['price_int'] <= max_p)).sum()
    pct = (count / len(df)) * 100
    print(f"  {label:15s}: {count:6,}개 ({pct:5.2f}%)")

# 5-2. 플랫폼별 게임 수
print("\n" + "="*80)
print("🖥️ 플랫폼별 게임 분포")
print("="*80)

platform_scenarios = {
    'Windows만': 0,
    'Windows + Mac': 0,
    'Windows + Linux': 0,
    'Windows + Mac + Linux': 0,
    'Mac 포함': 0,
    'Linux 포함': 0
}

for idx, row in df.iterrows():
    if isinstance(row['platforms'], dict):
        win = row['platforms'].get('windows', False)
        mac = row['platforms'].get('mac', False)
        linux = row['platforms'].get('linux', False)
        
        if win and not mac and not linux:
            platform_scenarios['Windows만'] += 1
        if win and mac and not linux:
            platform_scenarios['Windows + Mac'] += 1
        if win and not mac and linux:
            platform_scenarios['Windows + Linux'] += 1
        if win and mac and linux:
            platform_scenarios['Windows + Mac + Linux'] += 1
        if mac:
            platform_scenarios['Mac 포함'] += 1
        if linux:
            platform_scenarios['Linux 포함'] += 1

print(f"\n📊 플랫폼 조합별 게임 수:")
for scenario, count in platform_scenarios.items():
    pct = (count / len(df)) * 100
    print(f"  {scenario:25s}: {count:6,}개 ({pct:5.2f}%)")

# 5-3. 한국어 지원 + 가격 조합
print("\n" + "="*80)
print("🇰🇷 한국어 지원 게임 분석 (한국 유저 특화)")
print("="*80)

kr_supported = df[df['is_korean_supported'] == True]
print(f"\n✅ 한국어 자막 지원: {len(kr_supported):,}개 ({len(kr_supported)/len(df)*100:.2f}%)")

# 한국어 지원 + 가격대
print(f"\n📊 한국어 지원 게임의 가격 분포:")
kr_price_dist = [
    (0, 0, "무료"),
    (1, 10000, "1만원 이하"),
    (10001, 20000, "1-2만원"),
    (20001, 50000, "2-5만원"),
    (50001, float('inf'), "5만원 초과")
]

for min_p, max_p, label in kr_price_dist:
    count = ((kr_supported['price_int'] >= min_p) & 
             (kr_supported['price_int'] <= max_p)).sum()
    pct = (count / len(kr_supported)) * 100
    print(f"  {label:15s}: {count:5,}개 ({pct:5.2f}%)")

# 한국어 지원 + 인기도
kr_popular = kr_supported[kr_supported['recommendations_total'] >= 1000]
print(f"\n✅ 한국어 지원 + 인기게임(1K+ 추천): {len(kr_popular):,}개")

# 5-4. 멀티플레이어 게임 분석
print("\n" + "="*80)
print("👥 멀티플레이어 게임 분석")
print("="*80)

# 카테고리에서 멀티플레이어 관련 찾기
def has_multiplayer(categories):
    if not isinstance(categories, list):
        return False
    mp_keywords = ['Multi-player', 'Co-op', 'Online Co-op', 'Online PvP']
    return any(keyword in cat for cat in categories for keyword in mp_keywords)

df['is_multiplayer'] = df['categories_en'].apply(has_multiplayer)
mp_games = df[df['is_multiplayer'] == True]

print(f"\n✅ 멀티플레이어 게임: {len(mp_games):,}개 ({len(mp_games)/len(df)*100:.2f}%)")

# 멀티플레이어 + 무료
mp_free = mp_games[mp_games['is_free'] == True]
print(f"  └─ 무료 멀티플레이어: {len(mp_free):,}개")

# 멀티플레이어 + 한국어
mp_kr = mp_games[mp_games['is_korean_supported'] == True]
print(f"  └─ 한국어 멀티플레이어: {len(mp_kr):,}개")

# 5-5. 품질 필터 조합
print("\n" + "="*80)
print("⭐ 품질 필터 조합 분석")
print("="*80)

quality_scenarios = [
    ("전체 게임", df),
    ("메타크리틱 70점 이상", df[df['metacritic'] >= 70]),
    ("추천수 1,000개 이상", df[df['recommendations_total'] >= 1000]),
    ("추천수 5,000개 이상", df[df['recommendations_total'] >= 5000]),
    ("메타 70+ AND 추천 1K+", df[(df['metacritic'] >= 70) & (df['recommendations_total'] >= 1000)]),
]

print(f"\n📊 품질 필터별 게임 수:")
for label, subset in quality_scenarios:
    count = len(subset)
    pct = (count / len(df)) * 100
    print(f"  {label:30s}: {count:6,}개 ({pct:5.2f}%)")

# 5-6. 최종 추천 필터 조합 예시
print("\n" + "="*80)
print("🎯 실전 추천 시나리오 예시")
print("="*80)

scenarios = [
    {
        'name': '한국어 + 무료 + 인기게임',
        'filter': (df['is_korean_supported'] == True) & 
                 (df['is_free'] == True) & 
                 (df['recommendations_total'] >= 1000)
    },
    {
        'name': '한국어 + 1만원 이하 + 평점 좋은 게임',
        'filter': (df['is_korean_supported'] == True) & 
                 (df['price_int'] > 0) & 
                 (df['price_int'] <= 10000) & 
                 (df['metacritic'] >= 75)
    },
    {
        'name': '멀티플레이어 + 한국어 + 추천 많은 게임',
        'filter': (df['is_multiplayer'] == True) & 
                 (df['is_korean_supported'] == True) & 
                 (df['recommendations_total'] >= 5000)
    },
    {
        'name': 'Mac 지원 + 한국어 + 인디게임',
        'filter': (df['platforms'].apply(lambda x: isinstance(x, dict) and x.get('mac', False))) & 
                 (df['is_korean_supported'] == True) & 
                 (df['genres_en'].apply(lambda x: isinstance(x, list) and 'Indie' in x))
    },
    {
        'name': '프리미엄 게임(5만원+) + 메타 80+',
        'filter': (df['price_int'] >= 50000) & 
                 (df['metacritic'] >= 80)
    }
]

print(f"\n📊 실전 시나리오별 필터링 결과:")
for scenario in scenarios:
    filtered = df[scenario['filter']]
    count = len(filtered)
    pct = (count / len(df)) * 100
    print(f"\n  【{scenario['name']}】")
    print(f"    매칭 게임: {count:,}개 ({pct:.2f}%)")
    
    if count > 0:
        # Top 3 게임 표시
        top_3 = filtered.nlargest(3, 'recommendations_total')
        print(f"    Top 3:")
        for i, (idx, game) in enumerate(top_3.iterrows(), 1):
            name = game['name'][:40] if len(game['name']) > 40 else game['name']
            recs = game['recommendations_total']
            print(f"      {i}. {name} (추천 {recs:,}개)")

# 5-7. 최종 데이터 요약
print("\n\n" + "="*80)
print("📊 최종 데이터 분석 요약")
print("="*80)

summary = f"""
【데이터 개요】
  • 총 게임 수: {len(df):,}개
  • 실제 게임(game): {(df['type'] == 'game').sum():,}개 ({(df['type'] == 'game').sum()/len(df)*100:.1f}%)
  • 한국 스토어 접근 가능: {(df['is_available_in_kr'] == True).sum():,}개 ({(df['is_available_in_kr'] == True).sum()/len(df)*100:.1f}%)

【데이터 품질】
  • 태그 데이터 완성도: {((df['tags_en'].apply(lambda x: isinstance(x, list) and len(x) > 0)).sum()/len(df)*100):.1f}%
  • 장르 데이터 완성도: {((df['genres_en'].apply(lambda x: isinstance(x, list) and len(x) > 0)).sum()/len(df)*100):.1f}%
  • 영어 설명 완성도: {((df['desc_en_length'] > 0).sum()/len(df)*100):.1f}%
  • 한국어 설명 완성도: {((df['desc_kr_length'] > 50).sum()/len(df)*100):.1f}%

【가격 분포】
  • 무료 게임: {(df['is_free'] == True).sum():,}개 ({(df['is_free'] == True).sum()/len(df)*100:.1f}%)
  • 유료 게임: {(df['price_int'] > 0).sum():,}개 ({(df['price_int'] > 0).sum()/len(df)*100:.1f}%)
  • 평균 가격(유료): {df[df['price_int'] > 0]['price_int'].mean():,.0f}원
  • 중앙값 가격: {df[df['price_int'] > 0]['price_int'].median():,.0f}원

【품질 지표】
  • 메타크리틱 점수 보유: {(df['metacritic'] > 0).sum():,}개 ({(df['metacritic'] > 0).sum()/len(df)*100:.1f}%)
  • 평균 메타크리틱: {df[df['metacritic'] > 0]['metacritic'].mean():.1f}점
  • 추천수 보유: {(df['recommendations_total'] > 0).sum():,}개 ({(df['recommendations_total'] > 0).sum()/len(df)*100:.1f}%)
  • 중앙값 추천수: {df[df['recommendations_total'] > 0]['recommendations_total'].median():,.0f}개

【한국 사용자 특화】
  • 한국어 자막 지원: {(df['is_korean_supported'] == True).sum():,}개 ({(df['is_korean_supported'] == True).sum()/len(df)*100:.1f}%)
  • 한국어 더빙 지원: {(df['is_korean_dubbed'] == True).sum():,}개 ({(df['is_korean_dubbed'] == True).sum()/len(df)*100:.1f}%)
  • 한국어 + 인기(1K+): {len(kr_popular):,}개

【플랫폼 지원】
  • Windows: {platform_scenarios['Windows만'] + platform_scenarios['Windows + Mac'] + platform_scenarios['Windows + Linux'] + platform_scenarios['Windows + Mac + Linux']:,}개 ({((platform_scenarios['Windows만'] + platform_scenarios['Windows + Mac'] + platform_scenarios['Windows + Linux'] + platform_scenarios['Windows + Mac + Linux'])/len(df)*100):.1f}%)
  • Mac: {platform_scenarios['Mac 포함']:,}개 ({(platform_scenarios['Mac 포함']/len(df)*100):.1f}%)
  • Linux: {platform_scenarios['Linux 포함']:,}개 ({(platform_scenarios['Linux 포함']/len(df)*100):.1f}%)

【추천 시스템 전략】
  ✅ 임베딩 방법: Rich Method (Description + Genres + Tags15 + Categories5)
  ✅ 평균 토큰 수: 115 토큰 (BGE-M3 최적)
  ✅ 필터링 기준: 가격/언어/플랫폼/품질/멀티플레이어
  ✅ 기본 추천 기준: 추천수 1,000개 이상 (검증된 게임)
"""

print(summary)

print("\n" + "="*80)
print("🎉 전체 데이터 분석 완료!")
print("="*80)

print("""
다음 단계 추천:
  1. 임베딩 생성 (BGE-M3 모델 사용)
  2. pgvector 데이터베이스 구축
  3. Hybrid Search 구현 (벡터 검색 + 메타데이터 필터링)
  4. LangChain 기반 추천 챗봇 개발
""")




🎯 5. 추천 시나리오별 필터링 조건 & 최종 요약

💰 가격대별 게임 분포 (필터링 옵션 설계)

📊 가격 필터 옵션별 게임 수:
  무료             :  4,368개 (11.91%)
  5천원 이하         :  6,021개 (16.42%)
  1만원 이하         : 13,023개 (35.52%)
  2만원 이하         : 22,882개 (62.41%)
  5만원 이하         : 29,220개 (79.69%)
  5만원 초과         :    614개 ( 1.67%)

🖥️ 플랫폼별 게임 분포

📊 플랫폼 조합별 게임 수:
  Windows만                 : 25,211개 (68.76%)
  Windows + Mac            :  4,413개 (12.04%)
  Windows + Linux          :  1,118개 ( 3.05%)
  Windows + Mac + Linux    :  5,918개 (16.14%)
  Mac 포함                   : 10,335개 (28.19%)
  Linux 포함                 :  7,039개 (19.20%)

🇰🇷 한국어 지원 게임 분석 (한국 유저 특화)

✅ 한국어 자막 지원: 8,818개 (24.05%)

📊 한국어 지원 게임의 가격 분포:
  무료             : 1,007개 (11.42%)
  1만원 이하         : 2,168개 (24.59%)
  1-2만원          : 2,427개 (27.52%)
  2-5만원          : 2,427개 (27.52%)
  5만원 초과         :   377개 ( 4.28%)

✅ 한국어 지원 + 인기게임(1K+ 추천): 2,886개

👥 멀티플레이어 게임 분석

✅ 멀티플레이어 게임: 9,294개 (25.35%)
  └─ 무료 멀티플레이어: 1,580개
  └─ 한국어 멀티플레이어: 2,652개

⭐ 품질 필터 조합 분석

📊 품